# Proyecto ML - parte 1 - Envío 5

**Grupo 18**

Integrantes:
- Hugo Lenin Guzman Monsalve
- Jose Luis Parra Fernandez
- Julian Camilo Rivera Guzman
- Sergio Alejandro Torres Melendez

## Clasificación de reseñas con roles de opinión, negaciones y RidgeClassifier

Este notebook resuelve la competencia de clasificación de sentimiento de reseñas de productos en español. Cada reseña debe quedar como negativa, neutral o positiva y la métrica es el accuracy. Solo se usan técnicas clásicas de scikit-learn, es decir TF-IDF sobre palabras y caracteres y un clasificador lineal, sin redes neuronales, transformers ni embeddings preentrenados.

Lo que hace difícil la tarea es que muchas reseñas mezclan elogios y quejas. Contar palabras positivas y negativas no alcanza, porque el sentimiento global depende de cuál de las opiniones pesa más. Por eso el modelo no trata la reseña como una bolsa de palabras. La divide en cláusulas, detecta cuáles expresan una opinión, las ordena de la más reciente a la más antigua y representa cada una según el papel que juega. A esto se suma un tratamiento explícito de las negaciones y un `RidgeClassifier` como clasificador final.

El notebook sigue este recorrido. Primero se prepara el entorno y se leen los datos, luego se explica el procesamiento del texto y la representación, después se presenta el modelo con su validación cruzada y por último se generan las predicciones, se revisan los resultados y se listan los archivos de la entrega.

## Preparación del entorno y los datos

Gran parte de la lógica del modelo vive en el módulo `features_grupo18.py`, que tiene que estar en la misma carpeta que este notebook. Ahí se definen la separación en cláusulas, el detector de opiniones y el transformador `AnalizadorDeResenas`, que convierte el texto crudo en las distintas vistas con las que se alimentan los vectorizadores. Se dejó fuera del notebook para que este se concentre en las decisiones de modelado, y también por una razón práctica. Una clase definida dentro de un notebook no se puede volver a cargar desde un archivo `.joblib` en otra máquina, mientras que una clase importada desde un módulo sí.

El módulo usa las stop words y el stemmer de NLTK. Como carga las stop words en el momento de importarse, la descarga se hace antes del `import`. Solo es necesaria la primera vez que se ejecuta en un entorno nuevo. Después se leen las reseñas de entrenamiento.

In [8]:
import os
import sys

import joblib
import nltk
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# El módulo carga las stop words al importarse, así que la descarga va antes (solo hace falta la primera vez)
nltk.download('stopwords', quiet=True)

sys.path.insert(0, os.getcwd())
import features_grupo18 as F

df_train = pd.read_csv('train.csv')
y_train = df_train['label']
print(f'{len(df_train)} reseñas | clases: {y_train.value_counts().to_dict()}')

12000 reseñas | clases: {'positivo': 4212, 'negativo': 4212, 'neutral': 3576}


## 1. Procesamiento de texto

Un modelo de aprendizaje no puede trabajar con texto crudo, así que cada reseña se limpia, se estructura y se convierte en vectores numéricos. Antes de vectorizar, el procesamiento tiene cuatro partes. Se normaliza el texto, se separa en cláusulas, se detectan las cláusulas que realmente opinan y se tratan las negaciones. De todo esto se encarga `AnalizadorDeResenas`, que es el primer paso del pipeline, de modo que el modelo final recibe directamente texto crudo.

### 1.1 Normalización y separación en cláusulas

Cada reseña se pasa a minúsculas y se le quitan las tildes, porque en los datos se usan de forma inconsistente y "también" y "tambien" terminarían como palabras distintas. Las stop words pasan por la misma normalización para que coincidan con el texto. Se conservan las que cambian el sentido de una opinión, como "no", "nunca", "sin", "pero", "muy" o "poco". Luego se extrae la raíz de cada palabra con el stemmer Snowball en español.

Después la reseña se parte en cláusulas. Se corta al final de cada oración, en los punto y coma y en las comas que van antes de un conector. Los conectores pueden ser contrastivos ("sin embargo", "aunque", "pero", "eso sí", "aun así"), de resumen ("al final", "en resumen", "con todo") o secundarios ("de paso", "por cierto", "además"). Cortar así importa porque una misma oración puede unir dos opiniones opuestas, como en "la batería es excelente, pero la cámara decepcionó", y cada una necesita tratarse por separado.

### 1.2 Detección de cláusulas de opinión

No todas las cláusulas de una reseña opinan. Muchas hablan de logística, como el envío, el empaque o lo que trae la caja, y funcionan como relleno. Además, las palabras de opinión aparecen repartidas casi por igual entre reseñas positivas y negativas, así que ver "terrible" en una reseña no basta para saber su clase. Lo que importa es cuál de las opiniones es la que decide.

Para encontrar las cláusulas de opinión se entrena un detector con los propios datos. Las reseñas neutrales solo hablan de logística, por lo que todas sus cláusulas sirven como ejemplos limpios de relleno. Como ejemplos de opinión se toman las cláusulas de reseñas positivas o negativas que contienen vocabulario que nunca aparece en una reseña neutral. Con esos dos conjuntos se ajusta un clasificador lineal sobre TF-IDF de palabras y caracteres. Como aprende también la forma de las frases evaluativas, logra reconocer opiniones armadas con palabras comunes, como "cumple de sobra" o "se queda corta", que el vocabulario por sí solo dejaría pasar.

### 1.3 Roles y marcadores de discurso

Con las cláusulas de opinión identificadas, se numeran de atrás hacia adelante según qué tan recientes son. La última opinión de la reseña recibe el rol `O1`, la anterior `O2` y las demás `O3`, y las cláusulas que no opinan reciben el rol `F`, de relleno. Se ordena por recencia porque en los datos la opinión que llega después suele ser la que define el sentimiento global. En las mediciones del grupo sobre train, la última cláusula de opinión coincide con la etiqueta de la reseña en cerca del 91 % de los casos.

También se detectan marcadores de discurso dentro de cada cláusula y se agregan como tokens propios, por ejemplo `O1_MARC_resumen`. No todos los conectores pesan igual. Al medir pares de opiniones enfrentadas, la opinión posterior gana alrededor del 91 % de las veces, sube a cerca del 97 % cuando la introduce una expresión de resumen como "al final" o "con todo" y baja a cerca del 39 % cuando va con un marcador secundario como "de paso" o "por cierto". Estas reglas no se programan a mano. Los marcadores se entregan como tokens y es el clasificador el que aprende cuánto peso darles.

### 1.4 Tratamiento de negaciones

Una palabra como "bueno" cambia de sentido en "no es bueno", pero una representación que solo guarda la raíz dejaría ambas frases casi iguales. Para evitarlo se activa `usar_negaciones=True` en el analizador. Cuando aparece "no", "nunca", "jamás", "sin", "ni" o "tampoco", las palabras que siguen, una vez descartadas las stop words, reciben el prefijo `NOT_`, de modo que "no es bueno" produce tokens equivalentes a `no NOT_buen`. El stemming se aplica antes de formar el token.

El alcance de la negación es local. La marca se reinicia al encontrar un signo de puntuación o al empezar otra cláusula, así que en "no es bueno, pero funciona" la negación afecta solo a la primera valoración y no vuelve negativa la segunda. Los tokens con `NOT_` aparecen en las vistas que pasan por stemming, que se describen en la siguiente sección.

Para ver cómo queda una reseña después del procesamiento se aplica `preprocesar_clausula` a la primera reseña de train. Se le pasa el texto completo y no una sola cláusula, pero como cada signo de puntuación reinicia la negación, el resultado es el mismo que si se procesara oración por oración.

Esta reseña es neutral y no tiene ninguna negación, así que ningún token lleva el prefijo `NOT_`. Aun así sirve para ver el resto del proceso, que incluye minúsculas, tildes, stop words y stemming. Para ver el prefijo en acción basta con cambiar `ejemplo` por una reseña que contenga "no", "sin" o "nunca".

In [10]:
ejemplo = df_train['text'].iloc[0]
print('Texto original:\n', ejemplo)
print('\nTokens con negación dentro de la cláusula:\n', F.preprocesar_clausula(ejemplo, usar_negaciones=True))

Texto original:
 Lo pedí un martes por la noche y me llegó al día siguiente. Es el modelo en color negro, el básico de la línea. Lo pesé en la báscula de la cocina y marca algo más de medio kilo. Lo tengo en el escritorio del trabajo y por ahora nomás lo uso de vez en cuando. Viene con su cable y el manual en la caja.

Tokens con negación dentro de la cláusula:
 ['pedi', 'mart', 'noch', 'lleg', 'dia', 'siguient', 'model', 'color', 'negr', 'basic', 'line', 'pes', 'bascul', 'cocin', 'marc', 'mas', 'medi', 'kil', 'escritori', 'trabaj', 'ahor', 'nom', 'uso', 'vez', 'vien', 'cabl', 'manual', 'caj']


## 2. Representación del texto

Después del procesamiento, cada reseña queda descrita por varias vistas de texto y cada vista se convierte en vectores con TF-IDF. Este esquema da a cada término un peso que depende de cuántas veces aparece en la reseña y de qué tan poco común es en el resto del corpus. Todos los vectorizadores usan `sublinear_tf=True`, que reemplaza la frecuencia por su logaritmo para que una palabra muy repetida no domine el vector.

| Bloque | Vista | Vectorizador | Lleva tokens `NOT_` |
| --- | --- | --- | --- |
| `epl` | marcado por posición E/P/L | TF-IDF palabras (1,2) | sí |
| `roles` | marcado por rol O1/O2/O3/F + marcadores | TF-IDF palabras (1,2) | sí |
| `car_ult` | última cláusula del texto | TF-IDF caracteres `char_wb` (2,5) | no |
| `car_o1` | última cláusula de opinión | TF-IDF caracteres `char_wb` (2,5) | no |
| `pal_o1` | última cláusula de opinión | TF-IDF palabras (1,3) | no |

Cada bloque mira la reseña desde un ángulo distinto. En `epl` cada token lleva la posición de su cláusula dentro del texto, con E para las anteriores, P para la penúltima y L para la última. En `roles` el prefijo es el rol de opinión y además se incluyen los marcadores de discurso. Los otros tres bloques se concentran en las cláusulas que más pesan, que son la última del texto y la última de opinión. Si una reseña no tiene ninguna cláusula de opinión detectada, la vista de la última opinión usa su última cláusula.

Los bloques de caracteres toman n-gramas de 2 a 5 caracteres dentro de cada palabra. Sirven porque los datos tienen errores de ortografía como "encmia", "peero" o "taalle" que el stemming no corrige, y esos fragmentos siguen coincidiendo con los de la palabra bien escrita. El bloque `pal_o1` usa unigramas, bigramas y trigramas sin stemming, y por eso guarda combinaciones como "no funciona".

Los tokens `NOT_` solo aparecen en `epl` y `roles`, que son los que pasan por stemming. En los otros tres bloques la negación se captura de otra forma, porque los n-gramas de palabras de `pal_o1` guardan combinaciones como "no funciona" y los n-gramas de caracteres ayudan con variantes morfológicas, errores pequeños y fragmentos compartidos entre palabras.

El analizador calcula además otras columnas, como polaridades numéricas, conteos de cláusulas y la vista de las dos últimas opiniones, pero este modelo solo trabaja con las cinco vistas de texto de la tabla. El `ColumnTransformer` las vectoriza por separado y las une en una sola matriz dispersa.

In [9]:
def tfidf_marcado(**kwargs):
    """TF-IDF de palabras sobre tokens que llevan prefijo de rol o de posición."""
    params = {
        'ngram_range': (1, 2),
        'sublinear_tf': True,
        'min_df': 2,
        'token_pattern': r'\S+',
        'lowercase': False,
    }
    params.update(kwargs)
    return TfidfVectorizer(**params)


def tfidf_caracteres():
    """n-gramas de caracteres dentro de los límites de cada palabra."""
    return TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 5),
        min_df=3,
        sublinear_tf=True,
    )


representacion = ColumnTransformer([
    ('epl', tfidf_marcado(), 'epl'),
    ('roles', tfidf_marcado(), 'roles'),
    ('car_ult', tfidf_caracteres(), 'ultima'),
    ('car_o1', tfidf_caracteres(), 'o1'),
    ('pal_o1', TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, min_df=2), 'o1'),
])

## 3. Modelo

### 3.1 RidgeClassifier

El clasificador final es `RidgeClassifier`, un modelo lineal que aprende un peso por cada columna de la representación y predice la clase con el puntaje más alto. Lo particular es el criterio con el que se entrena. Convierte el problema en una regresión, asignando +1 a las reseñas de una clase y −1 al resto, y ajusta los pesos por mínimos cuadrados con una penalización L2 sobre su tamaño.

En la práctica esto hace que todas las reseñas participen en el ajuste, y no solo las que quedan cerca de la frontera, como ocurre con una SVM lineal. Además, la representación tiene muchas columnas parecidas entre sí, porque un mismo término puede aparecer en `epl`, en `roles` y en `pal_o1`, y la penalización L2 tiende a repartir el peso entre ellas en lugar de cargarlo todo en una sola.

El hiperparámetro `alpha` controla esa penalización y funciona al revés que el `C` de una regresión logística o de una SVM, porque un `alpha` más alto regulariza más. Se usa `alpha=1.0`, que es el valor por defecto de scikit-learn, y en este notebook no se exploran otros valores. Durante el desarrollo también se probó un `LinearSVC` con la misma representación y `RidgeClassifier` dio un accuracy ligeramente mejor, por eso quedó como modelo final. Esa comparación no se reproduce aquí. Igual que el `LinearSVC`, `RidgeClassifier` no entrega probabilidades sino solo puntajes por clase con `decision_function`, y para la métrica del proyecto no hacen falta.

### 3.2 Pipeline completo

Todo se junta en un único `Pipeline` de tres pasos. El primero es el analizador con las negaciones activadas, que convierte el texto crudo en las vistas descritas antes. El segundo es la representación TF-IDF por bloques y el tercero es el clasificador. Al ser un solo objeto, el vocabulario, los pesos IDF, el detector de opiniones y los coeficientes se ajustan siempre juntos y con los mismos datos, y el modelo guardado al final sirve directamente sobre texto nuevo.

In [ ]:
def construir_modelo(clasificador):
    """Pipeline completo: texto crudo -> vistas con negaciones -> TF-IDF -> clasificador."""
    return Pipeline([
        ('analizador', F.AnalizadorDeResenas(usar_negaciones=True)),
        ('representacion', representacion),
        ('clf', clasificador),
    ])


modelo_ridge = construir_modelo(RidgeClassifier(alpha=1.0))

### 3.3 Validación cruzada

Para estimar el desempeño en datos que el modelo no ha visto se usa validación cruzada estratificada de cinco particiones. En cada iteración se entrena con cuatro particiones y se valida con la restante, y se repite hasta que todas hayan hecho de validación. La estratificación mantiene una proporción parecida de positivo, negativo y neutral en cada fold, `shuffle=True` mezcla las filas antes de dividirlas y `random_state=42` hace que el experimento se pueda repetir con los mismos folds.

El pipeline completo se ajusta dentro de cada fold, lo que incluye el detector de opiniones, los vectorizadores TF-IDF y el clasificador. Así la reseña de validación no influye en el vocabulario ni en los parámetros aprendidos durante ese fold, y se evita la fuga de información. El argumento `n_jobs=-1` ejecuta los folds en paralelo con los núcleos disponibles y no cambia el resultado.

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_ridge = cross_val_score(modelo_ridge, df_train['text'], y_train, cv=cv, scoring='accuracy', n_jobs=-1)

print('RidgeClassifier por fold:', scores_ridge.round(4))
print(f'RidgeClassifier promedio: {scores_ridge.mean():.4f} (+/- {scores_ridge.std():.4f})')

RidgeClassifier por fold: [0.9062 0.9004 0.91   0.9117 0.9021]
RidgeClassifier promedio: 0.9061 (+/- 0.0044)


### 3.4 Reporte por clase (opcional)

El accuracy promedio no dice en qué clases se equivoca el modelo. Para verlo se puede generar el reporte de precisión, recall y F1 por clase con `cross_val_predict`, que obtiene la predicción de cada reseña con el modelo del fold donde quedó como validación. El inconveniente es que obliga a entrenar el pipeline cinco veces más, algo costoso con este detector. Por eso `GENERAR_REPORTE_CV` está en `False` durante la ejecución normal. Si se quiere revisar el reporte, se cambia a `True` y se vuelve a ejecutar la celda.

Esta celda también fija `modelo_ridge` como el modelo final que se entrena y se exporta en la siguiente sección.

In [13]:
mejor_modelo = modelo_ridge
nombre_mejor = 'ridge'
GENERAR_REPORTE_CV = False

print(f'Mejor modelo nuevo: {nombre_mejor}')
if GENERAR_REPORTE_CV:
    from sklearn.model_selection import cross_val_predict
    y_pred = cross_val_predict(mejor_modelo, df_train['text'], y_train, cv=cv, n_jobs=-1)
    print(classification_report(y_train, y_pred, digits=4))
else:
    print('Reporte CV omitido para evitar cinco entrenamientos adicionales.')

Mejor modelo nuevo: ridge
Reporte CV omitido para evitar cinco entrenamientos adicionales.


## 4. Generación de predicciones y archivos

Una vez validado el enfoque, el modelo se entrena de nuevo con el 100 % de `train.csv`, de manera que aproveche todas las reseñas disponibles y no se reserve ninguna para validación. Después se cargan `eval.csv` y `sample_submission.csv`. El pipeline transforma las reseñas de evaluación con exactamente las mismas reglas aprendidas en el entrenamiento y entrega una etiqueta para cada una, que se guarda en un CSV con la columna `answer` respetando el formato pedido.

Además se guarda el pipeline completo con `joblib`. Así el modelo se puede reutilizar sin reconstruir a mano el analizador, los vocabularios TF-IDF ni el clasificador.

In [14]:
mejor_modelo.fit(df_train['text'], y_train)

df_eval = pd.read_csv('eval.csv')
df_sub = pd.read_csv('sample_submission.csv')
df_sub['answer'] = mejor_modelo.predict(df_eval['text'])

archivo_salida = f'submission_{nombre_mejor}_negacion_v5_grupo18.csv'
df_sub.to_csv(archivo_salida, index=False)
joblib.dump(mejor_modelo, f'modelo_{nombre_mejor}_negacion_v5_grupo18.joblib')

print(f'Archivo generado: {archivo_salida}')
print('Distribución de predicciones:')
print(df_sub['answer'].value_counts(normalize=True).round(4))

Archivo generado: submission_ridge_negacion_v5_grupo18.csv
Distribución de predicciones:
answer
positivo    0.367
negativo    0.357
neutral     0.276
Name: proportion, dtype: float64


## 5. Resultados

La validación cruzada de cinco folds dio un accuracy promedio de **90,61 %**, con una desviación de ±0,44 %. Por fold, el accuracy estuvo entre 90,04 % y 91,17 %, así que el desempeño es bastante estable frente a la partición de los datos.

Sobre el conjunto de evaluación, el modelo predijo 36,7 % de reseñas positivas, 35,7 % negativas y 27,6 % neutrales. Es una distribución parecida a la de train, donde positivo y negativo tienen cada uno el 35,1 % de las reseñas y neutral el 29,8 %.

El archivo para Kaggle es `submission_ridge_negacion_v5_grupo18.csv` y el pipeline entrenado queda guardado en `modelo_ridge_negacion_v5_grupo18.joblib`.

### Conclusiones

El modelo llega a cerca de 90,6 % de accuracy en validación cruzada. La idea que sostiene el diseño es que el sentimiento de una reseña no sale de contar palabras sino de decidir qué opinión manda. Por eso la representación se organiza alrededor de las cláusulas de opinión, de su orden y de los marcadores que las acompañan, y no alrededor de la reseña completa.

Este notebook no reporta resultados por clase porque el reporte de la sección 3.4 viene desactivado. Por cómo están construidos los datos, lo esperable es que neutral sea la clase más fácil, ya que sus reseñas solo hablan de logística, y que los errores que quedan se concentren entre positivo y negativo, en reseñas que mezclan opiniones de las dos polaridades. Para comprobarlo basta con activar `GENERAR_REPORTE_CV`.

El tratamiento de negaciones también tiene limitaciones. Su alcance depende de la puntuación y no de la sintaxis, así que en «no solo es barato sino que funciona bien» quedan negadas también «funciona» y «bien», aunque la frase es positiva. Además, algunas negaciones no niegan una opinión. En «sin duda lo recomiendo», la palabra «sin» marca «recomiendo» como negado y el token resultante se parece al de «no lo recomiendo».

Como siguiente paso valdría la pena revisar a mano los errores de las reseñas con varias opiniones, para ver si hay algún patrón sobre cuál opinión manda que el modelo todavía no capture.

### Archivos de la entrega

Para la entrega en Bloque Neón se suben este notebook, el módulo `features_grupo18.py` y el archivo `modelo_ridge_negacion_v5_grupo18.joblib`. El módulo es necesario para volver a cargar el modelo, porque el `.joblib` guarda una instancia de `AnalizadorDeResenas` y `joblib.load` necesita importar esa clase desde `features_grupo18.py` para reconstruirla. Sin ese archivo la carga falla.

Para usar el modelo guardado basta con ejecutar `joblib.load('modelo_ridge_negacion_v5_grupo18.joblib')` desde una carpeta donde esté el módulo y llamar a `predict` con una lista de reseñas en texto crudo.

## Uso de herramientas de inteligencia artificial

El diseño del enfoque (detección de cláusulas de opinión, polaridad por cláusula, recencia de la opinión que manda) y la implementación del código fueron desarrollados por nosotros. Usamos un asistente de IA como apoyo en la redacción de las explicaciones del código (la documentación en celdas markdown y los comentarios/docstrings del módulo eatures_grupo18.py), para expresar con más claridad lo que el código ya hacía, y como guia en cuanto a que mejorar en los diferentes envios